In [ ]:
import gradio as gr
from openai import OpenAI
import json


BASE_URL = "http://localhost:11434/v1"
LOCAL_MODEL = "qwen2.5"



/Users/mukesh/Desktop/chat_ai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
client = OpenAI(api_key="not apksplicable", base_url=BASE_URL)

In [3]:
def process_history(history: list):
    return [{"role": h['role'], "content": h['content']} for h in history]


In [4]:
def fetch_price(item: str):
    if item == 'marie gold':
        return "10"
    elif item == 'Diary Milk':
        return "50"
    else:
        return "Price not available"

In [5]:
price_function = {
    "name": "fetch_price",
    "description": "Fetch price for the electronic product",
    "parameters": {
        "type": "object",
        "properties": {
            "product_name": {
                "type": "string",
                "description": "The price of the product that the customer enquired",
            },
            "unit": {
                "type": "string",
                "enum": ["rupees"],
                "description": "Currency unit"
            }
        },
        "required": ["product_name"],
        "additionalProperties": False
    }
}

tools = [{'role': 'function', 'function': price_function}]


In [6]:
def handle_tool_calls(message):
    response = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == 'fetch_price':
            arguments = json.loads(tool_call.function.arguments)
            product_name = arguments.get('product_name')
            price_details = fetch_price(product_name)
            response.append({'role': 'tool', 'content': price_details, 'tool_id': tool_call.id})
    return response


In [7]:
def chat(message, history):
    print(history)
    hist = process_history(history)
    system_prompt = [{"role": "system", "content": "You are assistant of a kirana store. List out all the products to customer but when enquired about the price, you let me know"}]
    messages = system_prompt + hist + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model=LOCAL_MODEL, messages=messages, tools=tools)
    print(response.choices[0])

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = client.chat.completions.create(model=LOCAL_MODEL, messages=messages, tools=tools)


    return response.choices[0].message.content


In [8]:
def render_ui():
    ui = gr.ChatInterface(fn=chat)
    ui.launch()


In [ ]:
if __name__ == "__main__":
    render_ui()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


[]
Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{"name": "list_products", "parameters": {}} \n\n(Note: list_products function is assumed to exist and can be called directly)', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))
